# Notebook 17: Label Noise Detection

## Why This Matters
Training on mislabeled data silently degrades model performance. A dataset with 10% label noise can reduce accuracy by 5-15% — more than many architecture improvements. The standard approach of manual review is expensive and doesn't scale. Embedding-based detection finds the likely mislabels automatically: if an image labeled "cat" sits close to all the "dog" images in CLIP space, that's a signal worth investigating.

This is the `label-noise` tool foundation.

## Structure
1. Types of label noise and their impact (narrative)
2. Embedding-based mislabel detection
3. Cross-model disagreement
4. Confident learning — when the model itself identifies likely mislabels
5. Building a ranked suspicious-sample list
6. Bridge to tabular representations

## What You'll Learn
- How to use CLIP/DINOv2 embeddings to detect label-embedding conflicts
- How cross-model disagreement signals likely mislabels
- How confident learning (cleanlab) formalizes the embedding-based intuition
- How to build a ranked mislabel candidate list for human review


## Background

### Why embedding proximity predicts label correctness

CLIP and DINOv2 were trained without your labels — they learned visual semantics from image-text pairs or self-supervision. If a sample labeled "cat" is embedding-space-close to all the "dog" samples, CLIP is telling you something: this image looks like a dog to a model that has never seen your labels. That's evidence the label might be wrong.

This isn't foolproof — CLIP can be wrong, and some images genuinely look like multiple classes. But at scale, embedding-label conflicts are the strongest automated signal for mislabel detection.

### Confident learning

The [cleanlab](https://github.com/cleanlab/cleanlab) library formalizes this with a statistical framework called confident learning (Northcutt et al., 2021):

1. Train a classifier on the (potentially noisy) labels — or use a pretrained model
2. Compute out-of-sample predicted probabilities via cross-validation
3. Estimate the joint distribution of noisy and true labels
4. Find samples where predicted class ≠ given label with high confidence

The result is a ranked list of likely mislabels, ordered by confidence.


In [ ]:
# %pip install -q cleanlab torch torchvision openai-clip numpy scikit-learn matplotlib

In [ ]:
import numpy as np
import torch
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
import clip
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from torch.utils.data import DataLoader

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()

CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
print(f"Ready on {device}")

## 1. Embedding-Based Mislabel Detection

In [ ]:
# Load CIFAR-10 and inject synthetic label noise for demonstration
from torch.utils.data import Dataset

class NoisyDataset(Dataset):
    """CIFAR-10 with controlled label noise."""
    def __init__(self, base_dataset, noise_rate=0.1, seed=42):
        self.data   = base_dataset
        self.labels = list(base_dataset.targets)  # copy
        self.noisy_indices = set()
        rng = np.random.default_rng(seed)
        n_noisy = int(len(self.labels) * noise_rate)
        flip_idx = rng.choice(len(self.labels), n_noisy, replace=False)
        for idx in flip_idx:
            old_label = self.labels[idx]
            new_label = rng.choice([l for l in range(10) if l != old_label])
            self.labels[idx] = int(new_label)
            self.noisy_indices.add(int(idx))

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx][0], self.labels[idx]


base_test = torchvision.datasets.CIFAR10("/tmp/cifar10", train=False, download=True,
                                          transform=preprocess)
noisy_ds  = NoisyDataset(base_test, noise_rate=0.10)
noisy_loader = DataLoader(noisy_ds, batch_size=256, shuffle=False, num_workers=0)

print(f"Dataset: {len(noisy_ds)} samples, {len(noisy_ds.noisy_indices)} injected noise labels ({10:.0f}%)")

In [ ]:
# Encode all images with CLIP
print("Extracting CLIP embeddings...")
all_embeddings, all_labels, all_indices = [], [], []
with torch.no_grad():
    for batch_idx, (imgs, labels) in enumerate(noisy_loader):
        feats = clip_model.encode_image(imgs.to(device))
        feats = feats / feats.norm(dim=-1, keepdim=True)
        all_embeddings.append(feats.cpu().numpy())
        all_labels.extend(labels if isinstance(labels, list) else labels.tolist())

embeddings = np.vstack(all_embeddings)
labels     = np.array(all_labels)
true_labels = np.array(base_test.targets)

print(f"Embeddings: {embeddings.shape}")

In [ ]:
# Detect mislabels: find samples where label conflicts with embedding neighborhood
def embedding_mislabel_score(embeddings, labels, k=10):
    """Score each sample by how much its label disagrees with its k nearest neighbors.
    
    Returns: array of scores in [0, 1] — higher means more likely mislabeled.
    """
    sims = cosine_similarity(embeddings)
    np.fill_diagonal(sims, -1)
    n_samples = len(labels)
    scores = np.zeros(n_samples)

    for i in range(n_samples):
        # K nearest neighbors
        nn_idx = np.argsort(sims[i])[::-1][:k]
        nn_labels = labels[nn_idx]
        # Fraction of neighbors with DIFFERENT label
        disagreement = np.mean(nn_labels != labels[i])
        scores[i] = disagreement

    return scores


print("Computing neighborhood disagreement scores...")
# Use a subset for speed
N_EVAL = 2000
emb_sub  = embeddings[:N_EVAL]
lab_sub  = labels[:N_EVAL]
true_sub = true_labels[:N_EVAL]
noisy_mask = np.array([i in noisy_ds.noisy_indices for i in range(N_EVAL)])

scores = embedding_mislabel_score(emb_sub, lab_sub, k=15)

# Evaluate detection quality: are high-score samples actually mislabeled?
from sklearn.metrics import average_precision_score
ap = average_precision_score(noisy_mask, scores)
print(f"Average precision (detect injected noise): {ap:.3f}")
print(f"Random baseline: {noisy_mask.mean():.3f}")
print()
print("Top-20 most suspicious samples:")
print(f"  {'Idx':>6}  {'Score':>7}  {'Given':>12}  {'True':>12}  {'Noisy?':>8}")
print("-" * 55)
top20 = np.argsort(scores)[::-1][:20]
for idx in top20:
    print(f"  {idx:>6}  {scores[idx]:>7.3f}  {CLASSES[lab_sub[idx]]:>12}  {CLASSES[true_sub[idx]]:>12}  {'YES' if noisy_mask[idx] else 'no':>8}")

## 2. Confident Learning with Cleanlab

In [ ]:
try:
    from cleanlab.filter import find_label_issues
    CLEANLAB_AVAILABLE = True
except ImportError:
    CLEANLAB_AVAILABLE = False
    print("cleanlab not installed — showing API only")

if CLEANLAB_AVAILABLE:
    # Train a classifier to get predicted probabilities
    sc  = StandardScaler()
    clf = LogisticRegression(max_iter=300, C=0.1, random_state=42, n_jobs=-1)

    from sklearn.model_selection import cross_val_predict
    pred_probs = cross_val_predict(clf, sc.fit_transform(emb_sub), lab_sub,
                                   cv=5, method='predict_proba')

    # Find label issues using confident learning
    label_issues = find_label_issues(labels=lab_sub, pred_probs=pred_probs,
                                     return_indices_ranked_by='self_confidence')

    # Evaluate against injected noise
    detected_noisy = noisy_mask[label_issues]
    precision = detected_noisy[:len(label_issues)//2].mean()
    recall    = detected_noisy.sum() / noisy_mask.sum()

    print(f"Cleanlab label issues found: {len(label_issues)}")
    print(f"Precision (top half): {precision:.3f}")
    print(f"Recall:               {recall:.3f}")
    print()
    print("Top flagged issues:")
    for idx in label_issues[:10]:
        print(f"  [{idx}] given={CLASSES[lab_sub[idx]]}, true={CLASSES[true_sub[idx]]}, "
              f"score={scores[idx]:.3f}, noisy={noisy_mask[idx]}")
else:
    print("""
# Cleanlab API (install with: pip install cleanlab)

from cleanlab.filter import find_label_issues
from sklearn.model_selection import cross_val_predict

# Get predicted probabilities from any classifier
pred_probs = cross_val_predict(clf, embeddings, labels, cv=5, method='predict_proba')

# Find label issues — returns ranked indices
label_issues = find_label_issues(
    labels=labels,
    pred_probs=pred_probs,
    return_indices_ranked_by='self_confidence'  # or 'normalized_margin'
)

# Review flagged samples
for idx in label_issues[:100]:
    print(f"Suspicious: idx={idx}, label={labels[idx]}, predicted={pred_probs[idx].argmax()}")
""")

## 3. Building a Review Queue

In production, the output is a ranked list for human review:

```python
suspicious = pd.DataFrame({
    'sample_id':    label_issues,
    'given_label':  [CLASSES[labels[i]] for i in label_issues],
    'predicted_label': [CLASSES[pred_probs[i].argmax()] for i in label_issues],
    'confidence':   [1 - pred_probs[i, labels[i]] for i in label_issues],
    'nn_disagreement': scores[label_issues],
})
suspicious.sort_values('confidence', ascending=False).to_csv('review_queue.csv')
```

Human reviewers work through the ranked list, correcting labels. The return on investment is high: fixing the top 5% most suspicious samples (where precision is highest) typically recovers most of the noise-induced accuracy loss.


## 4. Bridge to Tabular Representations

Label noise detection works because CLIP embeddings encode visual semantics that correlate with correct labels. For tabular data, there's no pretrained general model — you need to learn embeddings from the data itself.

Notebook 18 covers tabular representation learning: entity embeddings for categorical features, TabNet, and SAINT — methods for turning structured data into meaningful vector representations that support the same kinds of similarity search, clustering, and anomaly detection you've seen throughout this series.


## Summary

| Method | Data needed | Interpretability | Precision |
|--------|-------------|-----------------|-----------|
| Embedding neighborhood | Embeddings + labels | High (show neighbors) | Moderate |
| Cross-model disagreement | Multiple models' predictions | High | High |
| Confident learning (cleanlab) | Embeddings + labels + classifier probs | Medium | High |

**Rule of thumb:** run embedding neighborhood scoring first (cheap, no model needed), then confident learning for a ranked list, then human review of the top 5-10%.

**Next:** Notebook 18 — Tabular Representations. Entity embeddings, TabNet, and SAINT — representation learning for structured data.
